# RNN part 1

Простейшая rnn посильвольной генерации текста

In [1]:
import pandas as pd
import time

import torch

In [3]:
df = pd.read_csv('../datas/data.csv')
df.head()

,Unnamed: 0,id,episode_id,number,raw_text,timestamp_in_ms,speaking_line,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
0,0,10368,35,29,"Lisa Simpson: Maggie, look. What's that?",235000,True,9,5.0,Lisa Simpson,Simpson Home,"Maggie, look. What's that?",maggie look whats that,4.0
1,1,10369,35,30,Lisa Simpson: Lee-mur. Lee-mur.,237000,True,9,5.0,Lisa Simpson,Simpson Home,Lee-mur. Lee-mur.,lee-mur lee-mur,2.0
2,2,10370,35,31,Lisa Simpson: Zee-boo. Zee-boo.,239000,True,9,5.0,Lisa Simpson,Simpson Home,Zee-boo. Zee-boo.,zee-boo zee-boo,2.0
3,3,10372,35,33,Lisa Simpson: I'm trying to teach Maggie that ...,245000,True,9,5.0,Lisa Simpson,Simpson Home,I'm trying to teach Maggie that nature doesn't...,im trying to teach maggie that nature doesnt e...,24.0
4,4,10374,35,35,"Lisa Simpson: It's like an ox, only it has a h...",254000,True,9,5.0,Lisa Simpson,Simpson Home,"It's like an ox, only it has a hump and a dewl...",its like an ox only it has a hump and a dewlap...,18.0


In [5]:
phrases = df['normalized_text'].tolist()
phrases[:10]

['maggie look whats that',
 'lee-mur lee-mur',
 'zee-boo zee-boo',
 'im trying to teach maggie that nature doesnt end with the barnyard i want her to have all the advantages that i didnt have',
 'its like an ox only it has a hump and a dewlap hump and dew-lap hump and dew-lap',
 'you know his blood type how romantic',
 'oh yeah whats my shoe size',
 'ring',
 'yes dad',
 'ooh look maggie what is that do-dec-ah-edron dodecahedron']

In [8]:
text = [[c for c in ph] for ph in phrases if type(ph) is str]
text[1:2]

[['l', 'e', 'e', '-', 'm', 'u', 'r', ' ', 'l', 'e', 'e', '-', 'm', 'u', 'r']]

## Создаем массив с данными

1. Разбиваем данные на токены (в нашем случае - символы)
2. Кодируем токены числами
3. Превращаем в эмбеддинги

In [11]:
CHARS = set('abcdefghijklmnopqrstuvwxyz ')
INDEX_TO_CHAR = ['none'] + [w for w in CHARS]
CHAR_TO_INDEX = {w: i for i, w in enumerate(INDEX_TO_CHAR)}

In [12]:
len(INDEX_TO_CHAR)

28

In [16]:
CHAR_TO_INDEX

{'none': 0,
 ' ': 1,
 'o': 2,
 'g': 3,
 'f': 4,
 'j': 5,
 'y': 6,
 'b': 7,
 'h': 8,
 's': 9,
 'k': 10,
 'v': 11,
 'u': 12,
 'q': 13,
 'd': 14,
 'i': 15,
 't': 16,
 'z': 17,
 'w': 18,
 'a': 19,
 'm': 20,
 'x': 21,
 'p': 22,
 'c': 23,
 'l': 24,
 'e': 25,
 'r': 26,
 'n': 27}

In [20]:
MAX_LEN = 50 # ограничение длины батча
X = torch.zeros((len(text), MAX_LEN), dtype= int) # матрица, заполненная нулями, в которой будут размещены индексы токенов
for i in range(len(text)): # Для каждого предложения
    for j, w in enumerate(text[i]): # Для каждого токена
        if j >= MAX_LEN:
            break
        X[i, j] = CHAR_TO_INDEX.get(w, CHAR_TO_INDEX['none'])

In [21]:
X[0:5]

tensor([[20, 19,  3,  3, 15, 25,  1, 24,  2,  2, 10,  1, 18,  8, 19, 16,  9,  1,
         16,  8, 19, 16,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [24, 25, 25,  0, 20, 12, 26,  1, 24, 25, 25,  0, 20, 12, 26,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [17, 25, 25,  0,  7,  2,  2,  1, 17, 25, 25,  0,  7,  2,  2,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [15, 20,  1, 16, 26,  6, 15, 27,  3,  1, 16,  2,  1, 16, 25, 19, 23,  8,
          1, 20, 19,  3,  3, 15, 25,  1, 16,  8, 19, 16,  1, 27, 19, 16, 12, 26,
         25,  1, 14,  2, 25,  9, 27, 16,  1, 25, 27, 14,  1, 18],
        [15, 16,  9,  1, 24, 15, 10, 25,  1, 19, 27,  1,  2, 21,  1,  2, 27, 24,
       

## Embedding и RNN ячейки

In [22]:
len(text)

10891

In [23]:
X.shape

torch.Size([10891, 50])

In [26]:
embeddings = torch.nn.Embedding(len(INDEX_TO_CHAR), 28) # размер словаря, размер вектора для кодировки каждого слова
t = embeddings(X)
t.shape

torch.Size([10891, 50, 28])

In [28]:
t[:1,:,:]

tensor([[[ 0.7988, -0.5030, -0.5944,  ...,  1.8060,  1.4049,  1.4204],
         [ 0.3815, -1.5465,  1.0263,  ..., -2.8704,  0.0596,  0.1905],
         [-1.3926,  1.3741, -0.4186,  ...,  0.0149, -0.8160, -0.6995],
         ...,
         [ 0.0254,  0.5979,  1.2428,  ..., -1.3303, -1.5418, -1.5399],
         [ 0.0254,  0.5979,  1.2428,  ..., -1.3303, -1.5418, -1.5399],
         [ 0.0254,  0.5979,  1.2428,  ..., -1.3303, -1.5418, -1.5399]]],
       grad_fn=<SliceBackward0>)

In [29]:
rnn = torch.nn.RNN(28, 128, batch_first=True) # на вход: размер эмбеддинга, размер скрытого состояния и порядок размерностей
o, s = rnn(t[:5,:,:])
o.shape, s.shape

(torch.Size([5, 50, 128]), torch.Size([1, 5, 128]))

In [32]:
o, s2 = rnn(t[:5,:,:], s)
o.shape, s2.shape

(torch.Size([5, 50, 128]), torch.Size([1, 5, 128]))

## Реализация сети с RNN

3 слоя:
1. Embedding (30)
2. RNN (hidden_dim=128)
3. Полносвязный слой для предсказания буквы (28, то есть размер словаря)

In [81]:
class Network(torch.nn.Module):
    def __init__(self):
        super(Network, self).__init__()
        self.embedding = torch.nn.Embedding(28, 30)
        self.rnn = torch.nn.RNN(30, 128)
        self.out = torch.nn.Linear(128, 28)

    def forward(self, sentences, state=None):
        x = self.embedding(sentences)
        x, s = self.rnn(x)
        return self.out(x)    

In [82]:
model = Network()

In [83]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=.05)

Обучение:

In [84]:
for ep in range(50):
    start = time.time()
    train_loss = 0.
    train_passed = 0

    for i in range(int(len(X) / 100)):
        # размер батча = 100 элементов
        batch = X[i * 100:(i + 1) * 100]
        X_batch = batch[:, :-1]
        Y_batch = batch[:, 1:].flatten()

        optimizer.zero_grad()
        answers = model.forward(X_batch)
        answers = answers.view(-1, len(INDEX_TO_CHAR))
        loss = criterion(answers, Y_batch)
        train_loss += loss.item()

        loss.backward()
        optimizer.step()
        train_passed += 1

    print("Epoch {}, Time: {:.3f}, Train loss: {:.3f}".format(ep, time.time() - start, train_loss / train_passed))    



Epoch 0, Time: 0.722, Train loss: 2.140
Epoch 1, Time: 0.736, Train loss: 1.880
Epoch 2, Time: 0.728, Train loss: 1.825
Epoch 3, Time: 0.720, Train loss: 1.790
Epoch 4, Time: 0.712, Train loss: 1.766
Epoch 5, Time: 0.728, Train loss: 1.749
Epoch 6, Time: 0.723, Train loss: 1.737
Epoch 7, Time: 0.731, Train loss: 1.726
Epoch 8, Time: 0.722, Train loss: 1.718
Epoch 9, Time: 0.731, Train loss: 1.711
Epoch 10, Time: 0.729, Train loss: 1.705
Epoch 11, Time: 0.718, Train loss: 1.700
Epoch 12, Time: 0.713, Train loss: 1.695
Epoch 13, Time: 0.730, Train loss: 1.691
Epoch 14, Time: 0.716, Train loss: 1.687
Epoch 15, Time: 0.710, Train loss: 1.683
Epoch 16, Time: 0.724, Train loss: 1.680
Epoch 17, Time: 0.709, Train loss: 1.677
Epoch 18, Time: 0.716, Train loss: 1.674
Epoch 19, Time: 0.722, Train loss: 1.671
Epoch 20, Time: 0.701, Train loss: 1.668
Epoch 21, Time: 0.725, Train loss: 1.666
Epoch 22, Time: 0.712, Train loss: 1.664
Epoch 23, Time: 0.719, Train loss: 1.662
Epoch 24, Time: 0.719, Tra

## Генерация

1. Отправляем в модель буквы из предложения (прогревая состояние)
2. Берем самую вероятную букву и добавляем ее в предложение
3. Повторяем пока не получим none (0)

In [102]:
def generate_sentence(word):
    sentence = list(word)
    sentence = [CHAR_TO_INDEX.get(s, 0) for s in sentence]
    answer = model.forward(torch.tensor(sentence))
    probas, indices = answer.topk(1)
    return ''.join([INDEX_TO_CHAR[ind.item()] for ind in indices.flatten()])

In [103]:
generate_sentence('father')

' nhe e'

In [104]:
generate_sentence('It is')

'nonehtn '